<a href="https://colab.research.google.com/github/rocket-lab-dev/ARES-DBP/blob/main/code/Dataset_prepare_balanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Dependencies
This cell installs the `biopython` library, which is necessary for reading and parsing biological sequence data in FASTA format.

In [2]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.0 MB/s eta 0:00:00


This cell implements the ESM-2 150M offline feature extraction engine. It includes functions to parse FASTA files, extract features with dynamic padding on an A100 GPU, and save the features and aligned labels to disk. It handles Google Drive mounting, cache management, and model loading/unloading.

In [3]:
# -*- coding: utf-8 -*-
# 🚀 ESM-2 150M 离线特征全量提取引擎 (A100 极速优化版)
# 自动生成 Features (.npy) 和 完美对齐的 Labels (.npy)

import os
import gc
import torch
import numpy as np
from Bio import SeqIO
from tqdm.auto import tqdm
from transformers import AutoTokenizer, EsmModel
from google.colab import drive

# ==========================================
# 1. 路径与硬件配置
# ==========================================
print("🔄 正在挂载 Google Drive...")
drive.mount('/content/drive')

WORKSPACE_ROOT = "/content/drive/MyDrive"
DATA_ROOT = f"{WORKSPACE_ROOT}/Dataset"

# 🌟 新的输出特征目录
OUT_DIR = f"{WORKSPACE_ROOT}/cache-esm-feature/150m"
HF_CACHE = f"{WORKSPACE_ROOT}/cache_huggingface"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE

# 硬件加速
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# 模型配置
MODEL_ID = "facebook/esm2_t30_150M_UR50D"
BATCH_SIZE = 512  # A100 80GB 提取 150M 模型，可轻松开到 512
MAX_LEN = 1024

# ==========================================
# 2. 严格对齐的数据解析器
# ==========================================
def parse_fasta_strictly(file_path):
    """
    严格解析 FASTA，确保提取的序列完全符合下游网络要求。
    同步返回 序列(sequences) 和 标签(labels)，保证 100% 对齐。
    """
    print(f"📖 正在解析并过滤: {file_path}")
    sequences =[]
    labels =[]
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")

    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper().strip()
        # 严格过滤：长度在 50~1024 之间，且仅包含标准氨基酸
        if 50 <= len(seq) <= MAX_LEN and all(c in valid_aa for c in seq):
            try:
                # 兼容多种标签后缀格式
                id_str = str(record.id)
                if id_str.endswith('_1') or '|1' in id_str or 'pos' in id_str.lower():
                    label = 1
                elif id_str.endswith('_0') or '|0' in id_str or 'neg' in id_str.lower():
                    label = 0
                else:
                    label = int(id_str.split('_')[-1])

                sequences.append(seq)
                labels.append(label)
            except Exception:
                continue

    print(f"   ✅ 有效序列提取完毕: 共 {len(sequences)} 条 (已过滤非法及超长序列)")
    return sequences, np.array(labels, dtype=np.int64)

# ==========================================
# 3. 特征提取引擎 (A100 动态 Padding 极速版)
# ==========================================
@torch.inference_mode()
def extract_and_save_features(sequences, labels, prefix_name, model, tokenizer):
    feat_path = f"{OUT_DIR}/{prefix_name}_features.npy"
    label_path = f"{OUT_DIR}/{prefix_name}_labels.npy"

    if os.path.exists(feat_path) and os.path.exists(label_path):
        print(f"📦 目标文件 {feat_path} 已存在，跳过提取。")
        return

    all_feats =[]

    print(f"🚀 开始提取 {prefix_name} 特征 (Batch Size: {BATCH_SIZE})...")
    for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc=prefix_name):
        batch_seqs = sequences[i : i+BATCH_SIZE]

        # 动态 Padding (极大节省计算无效 token 的算力)
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        # 提取最后一层隐藏状态
        out = model(**inputs).last_hidden_state

        # 基于 Mask 的 Mean Pooling (去除 padding token 的干扰)
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

        all_feats.append(pooled.cpu().to(torch.float32).numpy())

    final_feats = np.concatenate(all_feats, axis=0)

    # 强制二次对齐校验
    assert len(final_feats) == len(labels), "❌ 严重错误：提取的特征数与标签数不对齐！"

    # 双双落盘
    np.save(feat_path, final_feats)
    np.save(label_path, labels)
    print(f"   💾 特征已保存 -> {feat_path} Shape: {final_feats.shape}")
    print(f"   💾 标签已保存 -> {label_path} Shape: {labels.shape}")

# ==========================================
# 4. 主执行流
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*80)
    print(f"🌟 ESM-2 150M 离线特征与对齐标签全自动提取 🌟")
    print("="*80)

    # 1. 加载模型
    print(f"\n🧠 正在将 {MODEL_ID} 加载至 A100 GPU...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
    model = EsmModel.from_pretrained(MODEL_ID, cache_dir=HF_CACHE, torch_dtype=torch.bfloat16).to("cuda")
    model.eval()

    # 2. 提取训练集
    print("\n" + "-"*50)
    train_seqs, train_labels = parse_fasta_strictly(f"{DATA_ROOT}/Train.fasta")
    extract_and_save_features(train_seqs, train_labels, "V2_150M_train", model, tokenizer)

    # 3. 提取测试集
    print("\n" + "-"*50)
    test_seqs, test_labels = parse_fasta_strictly(f"{DATA_ROOT}/Test.fasta")
    extract_and_save_features(test_seqs, test_labels, "V2_150M_test", model, tokenizer)

    # 显存清理
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    print("\n" + "🎉"*20)
    print(f"✨ 所有特征提取完毕！全部保存在: {OUT_DIR} ✨")
    print("🎉"*20)

🔄 正在挂载 Google Drive...
Mounted at /content/drive

🌟 ESM-2 150M 离线特征与对齐标签全自动提取 🌟

🧠 正在将 facebook/esm2_t30_150M_UR50D 加载至 A100 GPU...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--------------------------------------------------
📖 正在解析并过滤: /content/drive/MyDrive/Dataset/Train.fasta
   ✅ 有效序列提取完毕: 共 49623 条 (已过滤非法及超长序列)
🚀 开始提取 V2_150M_train 特征 (Batch Size: 512)...


V2_150M_train:   0%|          | 0/97 [00:00<?, ?it/s]

   💾 特征已保存 -> /content/drive/MyDrive/cache-esm-feature/150m/V2_150M_train_features.npy Shape: (49623, 640)
   💾 标签已保存 -> /content/drive/MyDrive/cache-esm-feature/150m/V2_150M_train_labels.npy Shape: (49623,)

--------------------------------------------------
📖 正在解析并过滤: /content/drive/MyDrive/Dataset/Test.fasta
   ✅ 有效序列提取完毕: 共 12543 条 (已过滤非法及超长序列)
🚀 开始提取 V2_150M_test 特征 (Batch Size: 512)...


V2_150M_test:   0%|          | 0/25 [00:00<?, ?it/s]

   💾 特征已保存 -> /content/drive/MyDrive/cache-esm-feature/150m/V2_150M_test_features.npy Shape: (12543, 640)
   💾 标签已保存 -> /content/drive/MyDrive/cache-esm-feature/150m/V2_150M_test_labels.npy Shape: (12543,)

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
✨ 所有特征提取完毕！全部保存在: /content/drive/MyDrive/cache-esm-feature/150m ✨
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉


### 2. Data Preparation and Balancing
This section mounts Google Drive, configures dataset paths, and defines a function to extract labels from FASTA headers. It also includes a pipeline to perform 1:1 under-sampling, ensuring the training data is balanced to prevent majority class bias.

This cell provides functionalities for data preparation and balancing of FASTA datasets. It mounts Google Drive, defines dataset paths, and includes a customizable `extract_label` function to parse labels from FASTA headers. The core logic performs 1:1 under-sampling to balance positive and negative classes, shuffles the data, and saves the balanced dataset to a new FASTA file, ensuring reproducible results through random seeding.

In [ ]:
import os
import random
from Bio import SeqIO
from google.colab import drive

# ==========================================
# 0. 环境与路径配置
# ==========================================
# 挂载 Google Drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Dataset"
TRAIN_FASTA = os.path.join(DATA_ROOT, "Train.fasta")
TEST_FASTA = os.path.join(DATA_ROOT, "Test.fasta")

BALANCED_TRAIN = os.path.join(DATA_ROOT, "Train_Balanced.fasta")
BALANCED_TEST = os.path.join(DATA_ROOT, "Test_Balanced.fasta")

# ==========================================
# 1. 标签提取器 (核心适配区)
# ==========================================
def extract_label(description: str) -> int:
    """
    【⚠️ 强制审查区】：根据你的 FASTA 头部格式提取 1(正类) 或 0(负类)。
    请根据你的实际数据，取消对应方案的注释，屏蔽其他方案。
    """
    desc_lower = description.lower()

    # 方案 A: 头部包含 "pos" 或 "neg" 关键字 (如 >Seq001 pos)
    # if 'pos' in desc_lower: return 1
    # if 'neg' in desc_lower: return 0

    # 方案 B: 头部以 |1 或 |0 结尾 (如 >P12345|1)
    # if description.strip().endswith('|1'): return 1
    # if description.strip().endswith('|0'): return 0

    # 方案 C: 头部 ID 包含特定标识 (如 >DBP_001 和 >NonDBP_001)
    # if 'non' in desc_lower: return 0
    # elif 'dbp' in desc_lower: return 1

    # 新方案 D: 头部包含 "_label_1" 或 "_label_0" (如 >seq_0_label_1)
    if '_label_1' in desc_lower: return 1
    if '_label_0' in desc_lower: return 0

    raise ValueError(f"无法从头部解析标签，请检查 FASTA 格式并修改提取逻辑: {description}")

# ==========================================
# 2. 核心采样管线与统计
# ==========================================
def balance_fasta_dataset(input_path: str, output_path: str, seed: int = 42):
    """
    读取 FASTA，统计样本，执行 1:1 欠采样，打乱后输出新 FASTA。
    """
    print("\n" + "="*50)
    print(f"🚀 开始处理: {os.path.basename(input_path)}")

    if not os.path.exists(input_path):
        print(f"❌ 错误: 找不到文件 {input_path}")
        return

    # 固定随机种子保证结果可复现
    random.seed(seed)

    pos_records = []
    neg_records =[]

    # 1. 内存全量读取与分类
    for record in SeqIO.parse(input_path, "fasta"):
        label = extract_label(record.description)
        if label == 1:
            pos_records.append(record)
        else:
            neg_records.append(record)

    # 2. 原始分布审计
    orig_pos_cnt = len(pos_records)
    orig_neg_cnt = len(neg_records)
    print(f"📊 [原始数据分布] 正样本(1): {orig_pos_cnt} | 负样本(0): {orig_neg_cnt}")

    if orig_pos_cnt == 0 or orig_neg_cnt == 0:
        print("❌ 错误: 某一类样本数量为 0，请检查 extract_label 解析逻辑！")
        return

    # 3. 动态寻找少数类边界 (Majority Class Collapse 防御)
    min_cnt = min(orig_pos_cnt, orig_neg_cnt)

    # 4. 多数类随机欠采样
    pos_sampled = random.sample(pos_records, min_cnt) if orig_pos_cnt > min_cnt else pos_records
    neg_sampled = random.sample(neg_records, min_cnt) if orig_neg_cnt > min_cnt else neg_records

    # 5. 合并与全局乱序 (Shuffle)
    # 乱序极其重要，防止模型在 DataLoader 顺序读取时发生灾难性遗忘
    balanced_records = pos_sampled + neg_sampled
    random.shuffle(balanced_records)

    # 6. 固化至本地
    SeqIO.write(balanced_records, output_path, "fasta")

    # 7. 结果校验
    print(f"⚖️  [平衡后分布] 正样本(1): {min_cnt} | 负样本(0): {min_cnt} | 总计: {min_cnt * 2}")
    print(f"✅ 文件已保存至: {output_path}")

# ==========================================
# 3. 执行流
# ==========================================
if __name__ == "__main__":
    # 执行训练集平衡
    balance_fasta_dataset(TRAIN_FASTA, BALANCED_TRAIN)

    # 执行测试集平衡
    # 审计提示：测试集通常需要保持原始分布以反映真实世界性能。
    # 仅当特定实验需求或指标对比要求时，才对测试集进行 1:1 平衡。
    balance_fasta_dataset(TEST_FASTA, BALANCED_TEST)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 开始处理: Train.fasta
📊 [原始数据分布] 正样本(1): 17857 | 负样本(0): 35428
⚖️  [平衡后分布] 正样本(1): 17857 | 负样本(0): 17857 | 总计: 35714
✅ 文件已保存至: /content/drive/MyDrive/Dataset/Train_Balanced.fasta

🚀 开始处理: Test.fasta
📊 [原始数据分布] 正样本(1): 4475 | 负样本(0): 8977
⚖️  [平衡后分布] 正样本(1): 4475 | 负样本(0): 4475 | 总计: 8950
✅ 文件已保存至: /content/drive/MyDrive/Dataset/Test_Balanced.fasta


### 3. ESM-2 Fine-tuning Pipeline (5-Fold CV)
This is the core training script. It configures the A100 GPU settings, loads the balanced FASTA data directly into RAM for speed, and constructs a LoRA-adapted ESM-2 (650M) model. It then runs a 5-Fold Cross-Validation process and saves the best model checkpoints.

This cell sets up the ESM-2 Fine-tuning Pipeline, optimized for A100 GPUs and large RAM. It configures hardware resources, loads balanced FASTA data into RAM, and defines the `ESM2_650M_SafeModel` architecture using LoRA for efficient fine-tuning. The script performs 5-Fold Stratified Cross-Validation, calculates various metrics (MCC, ACC, SN, SP, AUC), and saves the best model checkpoints for each fold. Finally, it evaluates the ensemble performance on the test set and saves detailed logs.

In [ ]:
# -*- coding: utf-8 -*-
import os
import gc
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

# 确保安装必要库
try:
    from Bio import SeqIO
except ImportError:
    !pip install -q biopython
    from Bio import SeqIO

# Install or upgrade torchao to a compatible version
# Peft library requires torchao >= 0.16.0
!pip install -q --upgrade torchao

# ==========================================
# 1. 顶级硬件资源配置 (A100 80GB + 230GB RAM)
# ==========================================
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Dataset"
SAVE_DIR = "/content/drive/MyDrive/Case_A_650M_Final_Optimized2"
HF_CACHE = "/content/drive/MyDrive/hf_cache"
LOCAL_CKPT = "/content/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(LOCAL_CKPT, exist_ok=True)

# os.environ['HF_HOME'] = HF_CACHE
# logging.set_verbosity_error()
# 强制禁用 Hugging Face 的 Token 自动获取机制
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1" # 顺便关闭数据收集以提升加载速度

# 650M 显存安全参数
MODEL_ID = "facebook/esm2_t33_650M_UR50D"
EMBED_DIM = 1280
BATCH_SIZE = 32   # A100 80GB 运行 650M+L1024 的推荐稳健值
MAX_LEN = 1024
LR = 5e-5
EPOCHS = 5
WARMUP_RATIO = 0.1

def clean_memory():
    """彻底回收显存防止 Fold 累积 OOM"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# ==========================================
# 2. 评估与数据处理
# ==========================================
def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "AUC": roc_auc_score(y_true, y_prob)
    }

def load_fasta_to_ram(file_path, tokenizer):
    sequences, labels = [], []
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"解析 FASTA: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            # 解析 >seq_X_label_Y
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"正在全量分词驻留 RAM (230GB 优势)...")
    encodings = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors="pt")
    return encodings['input_ids'], encodings['attention_mask'], torch.tensor(labels, dtype=torch.float)

class StaticRAMDataset(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

# ==========================================
# 3. 网络架构 (ESM-2 650M + LoRA + Checkpointing)
# ==========================================
class ESM2_650M_SafeModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = EsmModel.from_pretrained(
            MODEL_ID, cache_dir=HF_CACHE,
            torch_dtype=torch.bfloat16,
            device_map="cuda:0",
            attn_implementation="sdpa"
        )
        # 必须开启梯度检查点以处理 650M 处理 1024 长度时的激活值堆积
        base.gradient_checkpointing_enable()

        lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["query", "key", "value"], lora_dropout=0.05)
        self.encoder = get_peft_model(base, lora_cfg)
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM , 512),
            nn.BatchNorm1d(512),       # 强制动作 1：必须加入批量归一化防协变量偏移
            nn.GELU(),                 # 强制动作 2：使用 GELU 替代 ReLU
            nn.Dropout(0.3),           # 强制动作 3：强力的 Dropout (>=0.3)
            nn.Linear(512, 1)
        )
        # self.classifier = nn.Sequential(
        #     nn.Linear(EMBED_DIM, 512),
        #     nn.GELU(),
        #     nn.Linear(512, 1)
        # )

    def forward(self, ids, mask):
        out = self.encoder(ids, attention_mask=mask).last_hidden_state
        mask_f = mask.unsqueeze(-1).float()
        pooled = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)

# ==========================================
# 4. 训练流程 (5-Fold CV)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
f_ids, f_mask, f_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Train_Balanced.fasta"), tokenizer)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_process_records = []

for fold, (t_idx, v_idx) in enumerate(skf.split(f_ids, f_labels)):
    clean_memory()
    print(f"\n🚀 Fold {fold+1}/5 | 650M 显存安全模式运行")

    train_ld = DataLoader(StaticRAMDataset(f_ids[t_idx], f_mask[t_idx], f_labels[t_idx]), batch_size=BATCH_SIZE, shuffle=True)
    val_ld = DataLoader(StaticRAMDataset(f_ids[v_idx], f_mask[v_idx], f_labels[v_idx]), batch_size=BATCH_SIZE)

    model = ESM2_650M_SafeModel().to("cuda")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    total_steps = len(train_ld) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    loss_fn = nn.BCEWithLogitsLoss()
    best_fold_mcc = -1

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, labels in tqdm(train_ld, desc=f"Fold {fold+1} Ep {epoch+1}"):
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                loss = loss_fn(logits, labels.to("cuda"))
            loss.backward()
            optimizer.step()
            scheduler.step()

        # 验证
        model.eval()
        y_p, y_t = [], []
        with torch.no_grad():
            for ids, mask, labels in val_ld:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                y_p.extend(torch.sigmoid(logits).cpu().float().numpy())
                y_t.extend(labels.numpy())

        m = calculate_all_metrics(y_t, y_p)
        m.update({"Fold": fold+1, "Epoch": epoch+1})
        cv_process_records.append(m)
        print(f"-> Val MCC: {m['MCC']:.4f}")

        if m['MCC'] > best_fold_mcc:
            best_fold_mcc = m['MCC']
            save_path = f"{LOCAL_CKPT}/best_f650_fold{fold}.pt"
            torch.save(model.state_dict(), save_path)
            shutil.copy2(save_path, os.path.join(SAVE_DIR, f"best_f650_fold{fold}.pt"))

    del model, optimizer; clean_memory()

# 保存 CV 训练日志
pd.DataFrame(cv_process_records).to_csv(os.path.join(SAVE_DIR, "650m_cv_logs.csv"), index=False)

# ==========================================
# 5. Test.fasta 独立验证与集成 (Ensemble)
# ==========================================
print("\n🔥 开始 Test.fasta 独立验证与集成预测...")
t_ids, t_mask, t_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
test_ld = DataLoader(StaticRAMDataset(t_ids, t_mask, t_labels), batch_size=BATCH_SIZE)

all_fold_probs = []
independent_metrics_list = [] # 🌟 修正命名
y_true_test = t_labels.numpy()

for fold in range(5):
    clean_memory()
    print(f"推理中: Fold {fold}...")
    model = ESM2_650M_SafeModel().to("cuda")
    # 从本地缓存加载权重
    model.load_state_dict(torch.load(f"{LOCAL_CKPT}/best_f650_fold{fold}.pt"))
    model.eval()

    f_p = []
    with torch.no_grad():
        for ids, mask, _ in tqdm(test_ld, desc=f"Fold {fold} 推理"):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
            f_p.extend(torch.sigmoid(logits).cpu().float().numpy())

    all_fold_probs.append(f_p)

    # 记录该 Fold 单独表现
    m_fold = calculate_all_metrics(y_true_test, f_p)
    m_fold.update({"Fold": fold})
    independent_metrics_list.append(m_fold)
    del model; clean_memory()

# 计算集成 (Soft Voting)
ens_probs = np.mean(all_fold_probs, axis=0)
ens_metrics = calculate_all_metrics(y_true_test, ens_probs)
ens_metrics.update({"Fold": "Ensemble_Final"})

# 汇总表格并保存
final_report = pd.DataFrame(independent_metrics_list)
final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)
final_report.to_csv(os.path.join(SAVE_DIR, "650m_final_independent_test.csv"), index=False)

print(f"\n✅ 任务完成！结果已同步至 Drive: {SAVE_DIR}")
print(f"集成测试最终指标: MCC={ens_metrics['MCC']:.4f}, AUC={ens_metrics['AUC']:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
解析 FASTA: /content/drive/MyDrive/Dataset/Train_Balanced.fasta
正在全量分词驻留 RAM (230GB 优势)...

🚀 Fold 1/5 | 650M 显存安全模式运行


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 1 Ep 1:   0%|          | 0/818 [00:00<?, ?it/s]

Caching is incompatible with gradient checkpointing in EsmLayer. Setting `use_cache=False`.


...

-> Val MCC: 0.6310

🚀 Fold 2/5 | 650M 显存安全模式运行


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 2 Ep 1:   0%|          | 0/818 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 5. `Test.fasta` 独立验证与集成 (Ensemble) 代码

以下代码展示了如何加载训练好的 5-Fold 模型，对 `Test.fasta` 数据集进行推理，并计算最终的集成（Ensemble）预测结果和评估指标。请注意，这部分代码已在之前的运行中执行完毕，结果已保存到您的 Google Drive 中。

### 4. Independent Test & Ensemble Prediction
This standalone execution block loads the 5 trained model checkpoints from the CV phase, performs inference on the unseen `Test.fasta` dataset, and aggregates the results using a soft-voting ensemble method to calculate final metrics (MCC, AUC, etc.).

This cell provides a standalone execution block for independent testing and ensemble prediction using the previously trained 5-Fold models. It reloads the necessary components (data loading, model architecture, metrics calculation) and performs inference on the `Test.fasta` dataset. It then aggregates predictions from all folds using soft voting to compute final ensemble metrics, which are saved to a CSV file.

In [ ]:
from Bio import SeqIO
import torch
import numpy as np
import pandas as pd
import os
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix, roc_auc_score

MAX_LEN = 1024
BATCH_SIZE = 32


def load_fasta_to_ram(file_path, tokenizer):
    sequences, labels = [], []
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"解析 FASTA: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            # 解析 >seq_X_label_Y
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"正在全量分词驻留 RAM...")
    encodings = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors="pt")
    return encodings['input_ids'], encodings['attention_mask'], torch.tensor(labels, dtype=torch.float)

class StaticRAMDataset(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "AUC": roc_auc_score(y_true, y_prob)
    }

print("\n🔥 开始 Test.fasta 独立验证与集成预测...")
t_ids, t_mask, t_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
test_ld = DataLoader(StaticRAMDataset(t_ids, t_mask, t_labels), batch_size=BATCH_SIZE)

all_fold_probs = []
independent_metrics_list = [] # 🌟 修正命名
y_true_test = t_labels.numpy()

for fold in range(1):
    clean_memory()
    print(f"推理中: Fold {fold}...")
    model = ESM2_650M_SafeModel().to("cuda")
    # 从本地缓存加载权重
    model.load_state_dict(torch.load(f"{LOCAL_CKPT}/best_f650_fold{fold}.pt"))
    model.eval()

    f_p = []
    with torch.no_grad():
        for ids, mask, _ in tqdm(test_ld, desc=f"Fold {fold} 推理"):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
            f_p.extend(torch.sigmoid(logits).cpu().float().numpy())

    all_fold_probs.append(f_p)

    # 记录该 Fold 单独表现
    m_fold = calculate_all_metrics(y_true_test, f_p)
    m_fold.update({"Fold": fold})
    independent_metrics_list.append(m_fold)
    del model; clean_memory()

# 计算集成 (Soft Voting)
ens_probs = np.mean(all_fold_probs, axis=0)
ens_metrics = calculate_all_metrics(y_true_test, ens_probs)
ens_metrics.update({"Fold": "Ensemble_Final"})

# 汇总表格并保存
final_report = pd.DataFrame(independent_metrics_list)
final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)
final_report.to_csv(os.path.join(SAVE_DIR, "650m_final_independent_test.csv"), index=False)

print(f"\n✅ 任务完成！结果已同步至 Drive: {SAVE_DIR}")
print(f"集成测试最终指标: MCC={ens_metrics['MCC']:.4f}, AUC={ens_metrics['AUC']:.4f}")

### 5. Hyperparameter Optimization (Optuna)
This cell sets up an automated Hyperparameter Optimization (HPO) process using the `optuna` framework. It searches for the best combination of learning rate, LoRA configurations (r, alpha, dropout), and classifier dimensions to maximize the MCC score.

This cell implements an automated Hyperparameter Optimization (HPO) process using the Optuna framework, specifically for the ESM-2 650M model. It defines a search space for learning rate, LoRA configurations (r, alpha, dropout), and classifier dimensions. The objective function trains the model for a few epochs, evaluates its performance, and reports metrics to Optuna for pruning. The script includes optimizations for A100 GPUs, RAM-resident data, and dynamic batch padding to maximize search efficiency. Finally, it saves the best parameters and detailed trial logs.

In [ ]:
# -*- coding: utf-8 -*-
# ==========================================
# 硬件环境：Colab A100 (80GB VRAM) + 160GB RAM
# 核心架构：ESM-2 650M + LoRA + Multi-Pooling (GAP+GMP)
# 极限加速：160G RAM 驻留 + 动态 Padding + 异步预取 + 关闭梯度检查点
# 任务目标：Optuna 自动化超参数寻优 (HPO)
# ==========================================
import os
import gc
import builtins
import threading
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import optuna
from datetime import datetime
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, EsmModel, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split

# 确保安装必要库
try:
    from Bio import SeqIO
except ImportError:
    os.system("pip install -q biopython")
    from Bio import SeqIO

# ==========================================
# 🚀 0. A100 专属硬件级性能解锁
# ==========================================
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# ==========================================
# 1. 环境与路径配置
# ==========================================
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    builtins.print("🔄 正在挂载 Google Drive...")
    drive.mount('/content/drive')

WORKSPACE_ROOT = "/content/drive/MyDrive"
DATA_ROOT =  f"{WORKSPACE_ROOT}/Dataset"
PROJECT_ROOT = f"{WORKSPACE_ROOT}/Zero-shot-Project"

HF_CACHE_DIR = f"{WORKSPACE_ROOT}/cache_huggingface"
os.environ['HF_HOME'] = HF_CACHE_DIR
logging.set_verbosity_error()

for d in ["models", "results", "logs", "optuna"]:
    os.makedirs(f"{PROJECT_ROOT}/{d}", exist_ok=True)

LOG_FILE = f"{PROJECT_ROOT}/logs/V2_Optuna_HPO_650M_Fast_run.log"
log_lock = threading.Lock()

def dual_print(*args, **kwargs):
    with log_lock:
        builtins.print(*args, **kwargs)
        if kwargs.get("end") in ["", "\r"]: return
        msg = " ".join(map(str, args))
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(f"[{timestamp}] {msg}\n")
print = dual_print

print("\n" + "="*80)
print(f"🚀 新任务启动: [A100 满血版] ESM-2 650M LoRA 极速超参搜索 | 日志: {LOG_FILE}")
print("="*80)

# ==========================================
# 2. 数据引擎与预处理 (160G RAM 驻留 + 动态 Padding)
# ==========================================
class DataEngineV2:
    @staticmethod
    def parse_fasta(path):
        print(f"📖 读取 FASTA: {path}")
        results =[]
        valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
        for rec in SeqIO.parse(path, "fasta"):
            try:
                label = int(rec.id.split('_')[-1])
                seq = str(rec.seq).upper().strip()
                if 50 <= len(seq) <= 1024 and all(c in valid_aa for c in seq):
                    results.append({"sequence": seq, "label": label})
            except Exception: continue
        return pd.DataFrame(results)

def prepare_data_in_ram(df, model_id):
    """将数据 Tokenize 后驻留在 160GB RAM 中，不进行全局 Padding"""
    print(f"🧬 预处理序列并驻留系统 RAM (利用 160GB 内存优势)...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=HF_CACHE_DIR)

    # ⚠️ 核心优化：padding=False，保留原始长度，节省海量计算
    encodings = tokenizer(df['sequence'].tolist(), truncation=True, max_length=1024, padding=False)

    ids =[torch.tensor(x, dtype=torch.long) for x in encodings['input_ids']]
    masks =[torch.tensor(x, dtype=torch.long) for x in encodings['attention_mask']]
    labels = torch.tensor(df['label'].values, dtype=torch.float32)

    print(f"✅ 数据驻留 RAM 完毕! 样本数: {len(labels)}")
    return ids, masks, labels

class ESMDatasetRAM(Dataset):
    def __init__(self, ids, masks, labels):
        self.ids, self.masks, self.labels = ids, masks, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.ids[idx], self.masks[idx], self.labels[idx]

# ⚠️ 核心优化：动态 Batch 组装 (Smart Batching)
def dynamic_collate_fn(batch):
    ids = [item[0] for item in batch]
    masks = [item[1] for item in batch]
    labels = torch.stack([item[2] for item in batch])

    # 仅填充到当前 Batch 的最大长度 (ESM-2 的 pad_token_id 为 1)
    padded_ids = pad_sequence(ids, batch_first=True, padding_value=1)
    padded_masks = pad_sequence(masks, batch_first=True, padding_value=0)

    return padded_ids, padded_masks, labels

# ==========================================
# 3. 动态支持超参的 ESM-2 650M (A100 极速版)
# ==========================================
class ESM2_650M_OptunaModel(nn.Module):
    def __init__(self, model_id, lora_r, lora_alpha, lora_drop, clf_hidden_dim, clf_drop):
        super().__init__()

        base_model = EsmModel.from_pretrained(
            model_id, cache_dir=HF_CACHE_DIR,
            torch_dtype=torch.bfloat16,
            device_map="cuda:0",
            attn_implementation="sdpa"
        )
        # 🚀 核心优化：开启梯度检查点以防 OOM！
        base_model.gradient_checkpointing_enable()

        lora_cfg = LoraConfig(
            r=lora_r, lora_alpha=lora_alpha,
            target_modules=["query", "key", "value"], lora_dropout=lora_drop
        )
        self.encoder = get_peft_model(base_model, lora_cfg)

        # 🌟 核心优化：恢复 Multi-Pooling (GAP+GMP)，输入维度翻倍 (1280*2=2560)
        self.classifier = nn.Sequential(
            nn.Linear(1280 * 2, clf_hidden_dim),
            nn.BatchNorm1d(clf_hidden_dim),
            nn.GELU(),
            nn.Dropout(clf_drop),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, ids, mask):
        out = self.encoder(ids, attention_mask=mask).last_hidden_state
        mask_f = mask.unsqueeze(-1).float()

        # GAP
        gap = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        # GMP (必须用 -1e9 掩盖 Padding 区域)
        out_masked = out.masked_fill(mask_f == 0, -1e9)
        gmp = out_masked.max(dim=1)[0]

        # 拼接 Multi-Pooling
        pooled = torch.cat([gap, gmp], dim=1)
        return self.classifier(pooled).squeeze(-1)

def calculate_mcc(y_true, y_pred):
    return matthews_corrcoef(y_true, y_pred)

# ==========================================
# 4. Optuna 目标函数 (Objective Function)
# ==========================================
def objective(trial, t_ids, t_masks, t_lbls, v_ids, v_masks, v_lbls):
    # -----------------------------------------
    # A. 定义搜索空间
    # -----------------------------------------
    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    lora_r = trial.suggest_categorical("lora_r",[8, 16, 32])
    lora_alpha = trial.suggest_categorical("lora_alpha",[16, 32, 64])
    lora_drop = trial.suggest_float("lora_drop", 0.0, 0.1, step=0.05)
    clf_hidden_dim = trial.suggest_categorical("clf_hidden_dim",[256, 512, 1024])
    clf_drop = trial.suggest_float("clf_drop", 0.1, 0.5, step=0.1)

    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 🚀 启动 Trial {trial.number}")
    print(f"参数: LR={lr:.2e}, r={lora_r}, alpha={lora_alpha}, L-Drop={lora_drop:.2f}, Clf-Dim={clf_hidden_dim}, Clf-Drop={clf_drop:.2f}")

    # -----------------------------------------
    # B. 模型与优化器初始化
    # -----------------------------------------
    # 🚀 核心优化：降低 Batch Size 稍微减少显存使用，确保在80GB以下稳定运行
    BATCH_SIZE = 16
    EPOCHS = 10

    train_loader = DataLoader(
        ESMDatasetRAM(t_ids, t_masks, t_lbls),
        batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=dynamic_collate_fn,
        num_workers=4, pin_memory=True, prefetch_factor=2
    )
    val_loader = DataLoader(
        ESMDatasetRAM(v_ids, v_masks, v_lbls),
        batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=dynamic_collate_fn,
        num_workers=4, pin_memory=True
    )

    model = ESM2_650M_OptunaModel(
        model_id="facebook/esm2_t33_650M_UR50D",
        lora_r=lora_r, lora_alpha=lora_alpha, lora_drop=lora_drop,
        clf_hidden_dim=clf_hidden_dim, clf_drop=clf_drop
    ).to('cuda')

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4, fused=True)
    criterion = torch.nn.BCEWithLogitsLoss()

    best_val_mcc = -1.0

    # -----------------------------------------
    # C. 训练与剪枝循环
    # -----------------------------------------
    try:
        for epoch in range(1, EPOCHS + 1):
            model.train()
            for ids, masks, labels in train_loader:
                # 异步拷贝至 GPU
                ids = ids.to('cuda', non_blocking=True)
                masks = masks.to('cuda', non_blocking=True)
                labels = labels.to('cuda', non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    loss = criterion(model(ids, masks), labels)
                loss.backward()
                optimizer.step()

            model.eval()
            val_preds, val_targets = [],[]
            with torch.no_grad():
                for ids, masks, labels in val_loader:
                    ids = ids.to('cuda', non_blocking=True)
                    masks = masks.to('cuda', non_blocking=True)
                    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                        logits = model(ids, masks)
                    probs = torch.sigmoid(logits).cpu().numpy()
                    val_preds.extend(probs)
                    val_targets.extend(labels.numpy())

            val_preds = np.array(val_preds)
            val_pred_labels = (val_preds > 0.5).astype(int)
            val_mcc = calculate_mcc(val_targets, val_pred_labels)

            if val_mcc > best_val_mcc:
                best_val_mcc = val_mcc

            # 🧠 报告中间结果供 Optuna 进行剪枝
            trial.report(val_mcc, epoch)
            if trial.should_prune():
                print(f"✂️ Trial {trial.number} 在 Epoch {epoch} 被剪枝 (当前 MCC: {val_mcc:.4f})")
                raise optuna.exceptions.TrialPruned()

        print(f"🏁 Trial {trial.number} 完成 | 最高 MCC: {best_val_mcc:.4f}")
        return best_val_mcc

    finally:
        # ⚠️ 极其重要：每个 Trial 结束必须彻底清空显存
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()

# ==========================================
# 5. 主执行流 (Optuna Study)
# ==========================================
if __name__ == "__main__":
    MODEL_ID = "facebook/esm2_t33_650M_UR50D"

    print("\n" + "🚀"*20 + "\n阶段 1: 加载训练集并进行 80/20 切分\n" + "🚀"*20)
    engine = DataEngineV2()
    df_train_full = engine.parse_fasta(f"{DATA_ROOT}/Train.fasta")

    df_hpo_train, df_hpo_val = train_test_split(df_train_full, test_size=0.2, random_state=42, stratify=df_train_full['label'])
    print(f"✅ HPO 拆分完成: Train={len(df_hpo_train)}, Val={len(df_hpo_val)}")

    # 预加载到 160GB RAM 中
    train_ids, train_masks, train_lbls = prepare_data_in_ram(df_hpo_train, MODEL_ID)
    val_ids, val_masks, val_lbls = prepare_data_in_ram(df_hpo_val, MODEL_ID)

    print("\n" + "🚀"*20 + "\n阶段 2: 启动 Optuna 超参自动化搜索\n" + "🚀"*20)

    study = optuna.create_study(
        direction="maximize",
        study_name="ESM2_650M_HPO_Fast",
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=3)
    )

    N_TRIALS = 20
    # 传入内存驻留的数据
    study.optimize(lambda trial: objective(trial, train_ids, train_masks, train_lbls, val_ids, val_masks, val_lbls),
                   n_trials=N_TRIALS, gc_after_trial=True)

    # ==========================================
    # 阶段 3: 输出最终搜优报告
    # ==========================================
    print("\n" + "🏆"*20 + "\n✨ 超参搜索完成！Optuna 最佳结果 ✨\n" + "🏆"*20)
    best_trial = study.best_trial

    print(f"🥇 最高 MCC: {best_trial.value:.4f}")
    print("🥇 最优超参数组合:")
    for key, value in best_trial.params.items():
        print(f"    - {key}: {value}")

    df_trials = study.trials_dataframe()
    optuna_csv_path = f"{PROJECT_ROOT}/optuna/ESM2_650M_HPO_Fast_Results.csv"
    df_trials.to_csv(optuna_csv_path, index=False)
    print(f"\n📁 详细搜索日志已存入: {optuna_csv_path}")
    print(f"📜 主日志文件已更新: {LOG_FILE}")

    import json
    with open(f"{PROJECT_ROOT}/optuna/best_params_650M_Fast.json", "w") as f:
        json.dump(best_trial.params, f, indent=4)



🚀 新任务启动: [A100 满血版] ESM-2 650M LoRA 极速超参搜索 | 日志: /content/drive/MyDrive/Zero-shot-Project/logs/V2_Optuna_HPO_650M_Fast_run.log

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
阶段 1: 加载训练集并进行 80/20 切分
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
📖 读取 FASTA: /content/drive/MyDrive/Dataset/Train.fasta
✅ HPO 拆分完成: Train=39698, Val=9925
🧬 预处理序列并驻留系统 RAM (利用 160GB 内存优势)...
✅ 数据驻留 RAM 完毕! 样本数: 39698
🧬 预处理序列并驻留系统 RAM (利用 160GB 内存优势)...


[I 2026-04-29 10:28:46,462] A new study created in memory with name: ESM2_650M_HPO_Fast


✅ 数据驻留 RAM 完毕! 样本数: 9925

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
阶段 2: 启动 Optuna 超参自动化搜索
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

[10:28:46] 🚀 启动 Trial 0
参数: LR=3.32e-05, r=8, alpha=16, L-Drop=0.05, Clf-Dim=1024, Clf-Drop=0.40


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

### 6. Additional Training Execution (Run A)
This is an additional copy of the 5-Fold Cross-Validation training script, likely used for running a separate experiment or verifying reproducibility.

This cell is an exact copy of the main 5-Fold Cross-Validation training script, likely used for running a separate experiment (Run A) or verifying reproducibility under similar conditions. It performs model training, validation, and evaluation on the dataset, saving checkpoints and logs independently.

This cell is another exact copy of the main 5-Fold Cross-Validation training script, likely used for running a separate experiment (Run B) or verifying reproducibility under similar conditions. It performs model training, validation, and evaluation on the dataset, saving checkpoints and logs independently.

In [ ]:
# -*- coding: utf-8 -*-
import os
import gc
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

# 确保安装必要库
try:
    from Bio import SeqIO
except ImportError:
    !pip install -q biopython
    from Bio import SeqIO

# Install or upgrade torchao to a compatible version
# Peft library requires torchao >= 0.16.0
!pip install -q --upgrade torchao

# ==========================================
# 1. 顶级硬件资源配置 (A100 80GB + 230GB RAM)
# ==========================================
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Dataset"
SAVE_DIR = "/content/drive/MyDrive/Case_A_650M_Final_Optimized2"
HF_CACHE = "/content/drive/MyDrive/hf_cache"
LOCAL_CKPT = "/content/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(LOCAL_CKPT, exist_ok=True)

# os.environ['HF_HOME'] = HF_CACHE
# logging.set_verbosity_error()
# 强制禁用 Hugging Face 的 Token 自动获取机制
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1" # 顺便关闭数据收集以提升加载速度

# 650M 显存安全参数
MODEL_ID = "facebook/esm2_t33_650M_UR50D"
EMBED_DIM = 1280
BATCH_SIZE = 32   # A100 80GB 运行 650M+L1024 的推荐稳健值
MAX_LEN = 1024
LR = 5e-5
EPOCHS = 5
WARMUP_RATIO = 0.1

def clean_memory():
    """彻底回收显存防止 Fold 累积 OOM"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# ==========================================
# 2. 评估与数据处理
# ==========================================
def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "AUC": roc_auc_score(y_true, y_prob)
    }

def load_fasta_to_ram(file_path, tokenizer):
    sequences, labels = [], []
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"解析 FASTA: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            # 解析 >seq_X_label_Y
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"正在全量分词驻留 RAM (230GB 优势)...")
    encodings = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors="pt")
    return encodings['input_ids'], encodings['attention_mask'], torch.tensor(labels, dtype=torch.float)

class StaticRAMDataset(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

# ==========================================
# 3. 网络架构 (ESM-2 650M + LoRA + Checkpointing)
# ==========================================
class ESM2_650M_SafeModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = EsmModel.from_pretrained(
            MODEL_ID, cache_dir=HF_CACHE,
            torch_dtype=torch.bfloat16,
            device_map="cuda:0",
            attn_implementation="sdpa"
        )
        # 必须开启梯度检查点以处理 650M 处理 1024 长度时的激活值堆积
        base.gradient_checkpointing_enable()

        lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["query", "key", "value"], lora_dropout=0.05)
        self.encoder = get_peft_model(base, lora_cfg)
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM , 512),
            nn.BatchNorm1d(512),       # 强制动作 1：必须加入批量归一化防协变量偏移
            nn.GELU(),                 # 强制动作 2：使用 GELU 替代 ReLU
            nn.Dropout(0.3),           # 强制动作 3：强力的 Dropout (>=0.3)
            nn.Linear(512, 1)
        )
        # self.classifier = nn.Sequential(
        #     nn.Linear(EMBED_DIM, 512),
        #     nn.GELU(),
        #     nn.Linear(512, 1)
        # )

    def forward(self, ids, mask):
        out = self.encoder(ids, attention_mask=mask).last_hidden_state
        mask_f = mask.unsqueeze(-1).float()
        pooled = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)

# ==========================================
# 4. 训练流程 (5-Fold CV)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
f_ids, f_mask, f_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Train_Balanced.fasta"), tokenizer)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_process_records = []

for fold, (t_idx, v_idx) in enumerate(skf.split(f_ids, f_labels)):
    clean_memory()
    print(f"\n🚀 Fold {fold+1}/5 | 650M 显存安全模式运行")

    train_ld = DataLoader(StaticRAMDataset(f_ids[t_idx], f_mask[t_idx], f_labels[t_idx]), batch_size=BATCH_SIZE, shuffle=True)
    val_ld = DataLoader(StaticRAMDataset(f_ids[v_idx], f_mask[v_idx], f_labels[v_idx]), batch_size=BATCH_SIZE)

    model = ESM2_650M_SafeModel().to("cuda")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    total_steps = len(train_ld) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    loss_fn = nn.BCEWithLogitsLoss()
    best_fold_mcc = -1

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, labels in tqdm(train_ld, desc=f"Fold {fold+1} Ep {epoch+1}"):
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                loss = loss_fn(logits, labels.to("cuda"))
            loss.backward()
            optimizer.step()
            scheduler.step()

        # 验证
        model.eval()
        y_p, y_t = [], []
        with torch.no_grad():
            for ids, mask, labels in val_ld:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                y_p.extend(torch.sigmoid(logits).cpu().float().numpy())
                y_t.extend(labels.numpy())

        m = calculate_all_metrics(y_t, y_p)
        m.update({"Fold": fold+1, "Epoch": epoch+1})
        cv_process_records.append(m)
        print(f"-> Val MCC: {m['MCC']:.4f}")

        if m['MCC'] > best_fold_mcc:
            best_fold_mcc = m['MCC']
            save_path = f"{LOCAL_CKPT}/best_f650_fold{fold}.pt"
            torch.save(model.state_dict(), save_path)
            shutil.copy2(save_path, os.path.join(SAVE_DIR, f"best_f650_fold{fold}.pt"))

    del model, optimizer; clean_memory()

# 保存 CV 训练日志
pd.DataFrame(cv_process_records).to_csv(os.path.join(SAVE_DIR, "650m_cv_logs.csv"), index=False)

# ==========================================
# 5. Test.fasta 独立验证与集成 (Ensemble)
# ==========================================
print("\n🔥 开始 Test.fasta 独立验证与集成预测...")
t_ids, t_mask, t_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
test_ld = DataLoader(StaticRAMDataset(t_ids, t_mask, t_labels), batch_size=BATCH_SIZE)

all_fold_probs = []
independent_metrics_list = [] # 🌟 修正命名
y_true_test = t_labels.numpy()

for fold in range(5):
    clean_memory()
    print(f"推理中: Fold {fold}...")
    model = ESM2_650M_SafeModel().to("cuda")
    # 从本地缓存加载权重
    model.load_state_dict(torch.load(f"{LOCAL_CKPT}/best_f650_fold{fold}.pt"))
    model.eval()

    f_p = []
    with torch.no_grad():
        for ids, mask, _ in tqdm(test_ld, desc=f"Fold {fold} 推理"):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
            f_p.extend(torch.sigmoid(logits).cpu().float().numpy())

    all_fold_probs.append(f_p)

    # 记录该 Fold 单独表现
    m_fold = calculate_all_metrics(y_true_test, f_p)
    m_fold.update({"Fold": fold})
    independent_metrics_list.append(m_fold)
    del model; clean_memory()

# 计算集成 (Soft Voting)
ens_probs = np.mean(all_fold_probs, axis=0)
ens_metrics = calculate_all_metrics(y_true_test, ens_probs)
ens_metrics.update({"Fold": "Ensemble_Final"})

# 汇总表格并保存
final_report = pd.DataFrame(independent_metrics_list)
final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)
final_report.to_csv(os.path.join(SAVE_DIR, "650m_final_independent_test.csv"), index=False)

print(f"\n✅ 任务完成！结果已同步至 Drive: {SAVE_DIR}")
print(f"集成测试最终指标: MCC={ens_metrics['MCC']:.4f}, AUC={ens_metrics['AUC']:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
解析 FASTA: /content/drive/MyDrive/Dataset/Train_Balanced.fasta
正在全量分词驻留 RAM (230GB 优势)...

🚀 Fold 1/5 | 650M 显存安全模式运行


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 1 Ep 1:   0%|          | 0/818 [00:00<?, ?it/s]

Caching is incompatible with gradient checkpointing in EsmLayer. Setting `use_cache=False`.


...

-> Val MCC: 0.6310

🚀 Fold 2/5 | 650M 显存安全模式运行


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 2 Ep 1:   0%|          | 0/818 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# -*- coding: utf-8 -*-
import os
import gc
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

# 确保安装必要库
try:
    from Bio import SeqIO
except ImportError:
    !pip install -q biopython
    from Bio import SeqIO

# Install or upgrade torchao to a compatible version
# Peft library requires torchao >= 0.16.0
!pip install -q --upgrade torchao

# ==========================================
# 1. 顶级硬件资源配置 (A100 80GB + 230GB RAM)
# ==========================================
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Dataset"
SAVE_DIR = "/content/drive/MyDrive/Case_A_650M_Final_Optimized2"
HF_CACHE = "/content/drive/MyDrive/hf_cache"
LOCAL_CKPT = "/content/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(LOCAL_CKPT, exist_ok=True)

# os.environ['HF_HOME'] = HF_CACHE
# logging.set_verbosity_error()
# 强制禁用 Hugging Face 的 Token 自动获取机制
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1" # 顺便关闭数据收集以提升加载速度

# 650M 显存安全参数
MODEL_ID = "facebook/esm2_t33_650M_UR50D"
EMBED_DIM = 1280
BATCH_SIZE = 32   # A100 80GB 运行 650M+L1024 的推荐稳健值
MAX_LEN = 1024
LR = 5e-5
EPOCHS = 5
WARMUP_RATIO = 0.1

def clean_memory():
    """彻底回收显存防止 Fold 累积 OOM"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# ==========================================
# 2. 评估与数据处理
# ==========================================
def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "AUC": roc_auc_score(y_true, y_prob)
    }

def load_fasta_to_ram(file_path, tokenizer):
    sequences, labels = [], []
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"解析 FASTA: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            # 解析 >seq_X_label_Y
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"正在全量分词驻留 RAM (230GB 优势)...")
    encodings = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors="pt")
    return encodings['input_ids'], encodings['attention_mask'], torch.tensor(labels, dtype=torch.float)

class StaticRAMDataset(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

# ==========================================
# 3. 网络架构 (ESM-2 650M + LoRA + Checkpointing)
# ==========================================
class ESM2_650M_SafeModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = EsmModel.from_pretrained(
            MODEL_ID, cache_dir=HF_CACHE,
            torch_dtype=torch.bfloat16,
            device_map="cuda:0",
            attn_implementation="sdpa"
        )
        # 必须开启梯度检查点以处理 650M 处理 1024 长度时的激活值堆积
        base.gradient_checkpointing_enable()

        lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["query", "key", "value"], lora_dropout=0.05)
        self.encoder = get_peft_model(base, lora_cfg)
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM , 512),
            nn.BatchNorm1d(512),       # 强制动作 1：必须加入批量归一化防协变量偏移
            nn.GELU(),                 # 强制动作 2：使用 GELU 替代 ReLU
            nn.Dropout(0.3),           # 强制动作 3：强力的 Dropout (>=0.3)
            nn.Linear(512, 1)
        )
        # self.classifier = nn.Sequential(
        #     nn.Linear(EMBED_DIM, 512),
        #     nn.GELU(),
        #     nn.Linear(512, 1)
        # )

    def forward(self, ids, mask):
        out = self.encoder(ids, attention_mask=mask).last_hidden_state
        mask_f = mask.unsqueeze(-1).float()
        pooled = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)

# ==========================================
# 4. 训练流程 (5-Fold CV)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
f_ids, f_mask, f_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Train_Balanced.fasta"), tokenizer)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_process_records = []

for fold, (t_idx, v_idx) in enumerate(skf.split(f_ids, f_labels)):
    clean_memory()
    print(f"\n🚀 Fold {fold+1}/5 | 650M 显存安全模式运行")

    train_ld = DataLoader(StaticRAMDataset(f_ids[t_idx], f_mask[t_idx], f_labels[t_idx]), batch_size=BATCH_SIZE, shuffle=True)
    val_ld = DataLoader(StaticRAMDataset(f_ids[v_idx], f_mask[v_idx], f_labels[v_idx]), batch_size=BATCH_SIZE)

    model = ESM2_650M_SafeModel().to("cuda")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    total_steps = len(train_ld) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    loss_fn = nn.BCEWithLogitsLoss()
    best_fold_mcc = -1

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, labels in tqdm(train_ld, desc=f"Fold {fold+1} Ep {epoch+1}"):
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                loss = loss_fn(logits, labels.to("cuda"))
            loss.backward()
            optimizer.step()
            scheduler.step()

        # 验证
        model.eval()
        y_p, y_t = [], []
        with torch.no_grad():
            for ids, mask, labels in val_ld:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                y_p.extend(torch.sigmoid(logits).cpu().float().numpy())
                y_t.extend(labels.numpy())

        m = calculate_all_metrics(y_t, y_p)
        m.update({"Fold": fold+1, "Epoch": epoch+1})
        cv_process_records.append(m)
        print(f"-> Val MCC: {m['MCC']:.4f}")

        if m['MCC'] > best_fold_mcc:
            best_fold_mcc = m['MCC']
            save_path = f"{LOCAL_CKPT}/best_f650_fold{fold}.pt"
            torch.save(model.state_dict(), save_path)
            shutil.copy2(save_path, os.path.join(SAVE_DIR, f"best_f650_fold{fold}.pt"))

    del model, optimizer; clean_memory()

# 保存 CV 训练日志
pd.DataFrame(cv_process_records).to_csv(os.path.join(SAVE_DIR, "650m_cv_logs.csv"), index=False)

# ==========================================
# 5. Test.fasta 独立验证与集成 (Ensemble)
# ==========================================
print("\n🔥 开始 Test.fasta 独立验证与集成预测...")
t_ids, t_mask, t_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
test_ld = DataLoader(StaticRAMDataset(t_ids, t_mask, t_labels), batch_size=BATCH_SIZE)

all_fold_probs = []
independent_metrics_list = [] # 🌟 修正命名
y_true_test = t_labels.numpy()

for fold in range(5):
    clean_memory()
    print(f"推理中: Fold {fold}...")
    model = ESM2_650M_SafeModel().to("cuda")
    # 从本地缓存加载权重
    model.load_state_dict(torch.load(f"{LOCAL_CKPT}/best_f650_fold{fold}.pt"))
    model.eval()

    f_p = []
    with torch.no_grad():
        for ids, mask, _ in tqdm(test_ld, desc=f"Fold {fold} 推理"):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
            f_p.extend(torch.sigmoid(logits).cpu().float().numpy())

    all_fold_probs.append(f_p)

    # 记录该 Fold 单独表现
    m_fold = calculate_all_metrics(y_true_test, f_p)
    m_fold.update({"Fold": fold})
    independent_metrics_list.append(m_fold)
    del model; clean_memory()

# 计算集成 (Soft Voting)
ens_probs = np.mean(all_fold_probs, axis=0)
ens_metrics = calculate_all_metrics(y_true_test, ens_probs)
ens_metrics.update({"Fold": "Ensemble_Final"})

# 汇总表格并保存
final_report = pd.DataFrame(independent_metrics_list)
final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)
final_report.to_csv(os.path.join(SAVE_DIR, "650m_final_independent_test.csv"), index=False)

print(f"\n✅ 任务完成！结果已同步至 Drive: {SAVE_DIR}")
print(f"集成测试最终指标: MCC={ens_metrics['MCC']:.4f}, AUC={ens_metrics['AUC']:.4f}")

### 7. Additional Training Execution (Run B)
Another instance of the ESM-2 training pipeline.